#  ⭐ `dim_soc_llt`

In [1]:
%run ../../config/bootstrap.py

import pandas as pd
from pathlib import Path
from utils import get_project_root

In [2]:
project_root = get_project_root()

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
base = (
    project_root
    / "data/01_bronze/meddra/MedDRA_28_1_Brazilian_Portuguese/ascii-281"
)

In [4]:
def read_meddra_ascii(file_path):
    df = pd.read_csv(
        file_path,
        sep=r"\$",
        header=None,
        dtype=str,
        encoding="latin-1",
        engine="python"
    )
    df = df.dropna(axis=1, how="all")
    return df

# 🥈 Silver

normalized data, medium quality

### MedDra

In [5]:
mdhier = read_meddra_ascii(base / "mdhier.asc")

mdhier.columns = [
    "PT_CODE",
    "HLT_CODE",
    "HLGT_CODE",
    "SOC_CODE",
    "PT_NAME",
    "HLT_NAME",
    "HLGT_NAME",
    "SOC_NAME",
    "SOC_ABBREV",
    "PRIMARY_SOC",
    "PRIMARY_FLAG"
]

mdhier.head()

,PT_CODE,HLT_CODE,HLGT_CODE,SOC_CODE,PT_NAME,HLT_NAME,HLGT_NAME,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,PRIMARY_FLAG
0,10002043,10002042,10002086,10005329,Anemia por deficiência de folato,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
1,10002080,10002042,10002086,10005329,Anemia por carência de vitamina B12,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
2,10002081,10002042,10002086,10005329,Anemia por carência de vitamina B6,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
3,10022972,10002042,10002086,10005329,Anemia ferropriva,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y
4,10034695,10002042,10002086,10005329,Anemia perniciosa,Anemiais carenciais,Anemias não hemolíticas e depressão medular,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,Y


In [13]:
llt = read_meddra_ascii(base / "llt.asc")
llt.columns = ["LLT_CODE", "LLT_NAME", "PT_CODE", "LLT_CURRENCY"]

hierarquia_completa = mdhier.merge(
    llt[["LLT_CODE", "LLT_NAME", "PT_CODE"]],
    on="PT_CODE",
    how="left"
)
hierarquia_completa['REACAO_CHAVE'] = hierarquia_completa.apply(
    lambda r: f"SOC_{r['SOC_CODE']}_HLGT_{r['HLGT_CODE']}_HLT_{r['HLT_CODE']}_PT_{r['PT_CODE']}_LLT_{r['LLT_CODE']}", 
    axis=1
) 
cols = ['REACAO_CHAVE','PRIMARY_FLAG','SOC_CODE','SOC_NAME', 'SOC_ABBREV', 'PRIMARY_SOC','HLGT_CODE','HLGT_NAME', 'HLT_CODE', 'HLT_NAME','PT_CODE', 'PT_NAME','LLT_CODE', 'LLT_NAME']


hierarquia_completa = hierarquia_completa[cols]
                      
hierarquia_completa.head()

,REACAO_CHAVE,PRIMARY_FLAG,SOC_CODE,SOC_NAME,SOC_ABBREV,PRIMARY_SOC,HLGT_CODE,HLGT_NAME,HLT_CODE,HLT_NAME,PT_CODE,PT_NAME,LLT_CODE,LLT_NAME
0,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002043,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002043,Anemia por deficiência de folato
1,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002044,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002044,Anemia por deficiência de ácido fólico
2,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002281,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002281,Anemia por carência de folato
3,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10002282,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10002282,Anemia por carência de ácido fólico
4,SOC_10005329_HLGT_10002086_HLT_10002042_PT_10002043_LLT_10016881,Y,10005329,Distúrbios dos sistemas hematológico e linfático,Blood,10005329,10002086,Anemias não hemolíticas e depressão medular,10002042,Anemiais carenciais,10002043,Anemia por deficiência de folato,10016881,Anemia por carência de folato


In [14]:
hierarquia_completa.shape

(142785, 14)

In [16]:
hierarquia_completa.REACAO_CHAVE.nunique()

142785

# 🥇 Gold

In [17]:
gold_path = (
    project_root
    / "data/03_gold/dim_soc_llt"
)

gold_path.mkdir(parents=True, exist_ok=True)

arquivo_saida = gold_path / "dim_soc_llt.parquet"

In [18]:
hierarquia_completa.to_parquet(
    arquivo_saida,
    index=False,
    engine="pyarrow"
)